# 🔐 CodeReviewEnv v3 — GRPO Training
**Train Qwen3-8B to investigate CVE vulnerabilities using MCP tools.**

⚡ **Before running:** Go to Runtime → Change runtime type → Select **T4 GPU**

In [ ]:
#@title Step 1: Upload & Extract Project (click the ▶ button)
from google.colab import files
import zipfile, os

print('📁 Upload Meta-project.zip when prompted...')
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/')
        print(f'✅ Extracted {fname}')

# Find the project directory
if os.path.exists('/content/Meta-project'):
    os.chdir('/content/Meta-project')
    print(f'✅ Working directory: {os.getcwd()}')
    print(f'✅ Files: {os.listdir(".")}')
else:
    print('❌ Meta-project folder not found. Check your zip file.')

In [ ]:
#@title Step 2: Install Dependencies (takes 3-5 minutes)
%%capture install_output
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.17" peft accelerate bitsandbytes
!pip install openenv-core fastmcp datasets matplotlib
print('✅ All dependencies installed!')

In [ ]:
#@title Step 3: Verify Everything Works
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/Meta-project')

from code_review_env.server.environment import CodeReviewEnvironment
env = CodeReviewEnvironment()
obs = env.reset(difficulty='easy')
print(f'✅ Environment loaded!')
print(f'✅ Ready to train!')

In [ ]:
#@title Step 4: 🚀 START TRAINING (3-5 hours)
# This is the main training cell.
# Once you press play, DO NOT close this tab.
# Go write your blog post while this runs.

import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/Meta-project')
os.chdir('/content/Meta-project')

%run train_grpo.py

In [ ]:
#@title Step 5: View Training Results
from IPython.display import Image, display
import json

# Show training curves
display(Image('grpo_output/training_curves.png', width=900))

# Show stats
with open('grpo_output/training_stats.json') as f:
    stats = json.load(f)
print(f'📊 Training Results:')
print(f'   Early mean reward: {stats["early_mean"]:.3f}')
print(f'   Late mean reward:  {stats["late_mean"]:.3f}')
print(f'   Improvement:       {stats["improvement"]:+.3f}')
print(f'   Max reward:        {stats["max_reward"]:.3f}')

In [ ]:
#@title Step 6: Download Results
from google.colab import files
import shutil

# Zip the output
shutil.make_archive('/content/grpo_results', 'zip', '/content/Meta-project/grpo_output')
files.download('/content/grpo_results.zip')
print('✅ Download started! Save this zip file.')